# 01. Initial Data Exploration — DGCP Public Procurement

**Objective:** profile the source dataset, identify data-quality issues, apply the same cleaning function used by the ETL, compare raw vs. cleaned data, and calculate the descriptive metrics that feed the reporting layer.

**Source:** Dominican Republic public procurement open data (DGCP), downloaded from `datos.gob.do`.

> In a fresh clone, the notebook uses `data/sample/dgcp_secp_sample_1000.csv` when the full raw CSV is not present. With the official CSV under `data/raw/`, the notebook reproduces the full-dataset analysis. The notebook creates `df_cleaned` in memory; `src/etl.py` is the script that persists a cleaned CSV for downstream SQL loading.

In [1]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from etl import clean_procurement_data

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)
print(f"Project root: {PROJECT_ROOT}")

Project root: C:\Users\User\Desktop\dgcp-public-procurement-analysis


In [2]:
RAW_PATH = PROJECT_ROOT / "data" / "raw" / "dgcp_secp_2015_2026.csv"
SAMPLE_PATH = PROJECT_ROOT / "data" / "sample" / "dgcp_secp_sample_1000.csv"
PROCESSED_PATH = PROJECT_ROOT / "data" / "processed" / "dgcp_secp_cleaned.csv"
DATA_PATH = RAW_PATH if RAW_PATH.exists() else SAMPLE_PATH

if not DATA_PATH.exists():
    raise FileNotFoundError(
        "No dataset found. Add the official CSV to data/raw/ or use the repository sample."
    )

df = pd.read_csv(DATA_PATH, low_memory=False)
print(f"Using dataset: {DATA_PATH.relative_to(PROJECT_ROOT)}")
print(f"Records: {len(df):,}")
print(f"Columns: {len(df.columns)}")
if DATA_PATH == SAMPLE_PATH:
    print(
        "Note: the repository sample is being used because the full raw CSV was not found. "
        "The sample is provided for reproducibility; anomaly counts and business metrics may differ from the full dataset."
    )


Using dataset: data\raw\dgcp_secp_2015_2026.csv
Records: 610,416
Columns: 18


In [3]:
display(df.head())
print("Column names:")
for i, col in enumerate(df.columns, 1):
    print(f"{i}. {col}")

,CODIGO_PROCESO,CODIGO_UNIDAD_COMPRA,UNIDAD_COMPRA,MODALIDAD,TIPO_EXCEPCION,CARATULA,ESTADO_PROCESO,MONEDA,MONTO_ESTIMADO,FECHA_PUBLICACION,HORA_PUBLICACION,DIRIGIDO_MIPYMES,DIRIGIDO_MIPYMES_MUJERES,OBJETO_PROCESO,DECRETO_PRESIDENCIAL,COMPRA_VERDE,COMPRA_CONJUNTA,URL
0,911-DAF-CM-2026-0012,887,Sistema Nacional de Atención a Emergencias y Seguridad 9-1-1,Contratación Menor,Proceso ordinario,Proceso dirigido a MIPYMES para el alquiler de furgones para oficina y almacén del Sistema Nacional de Atención a Emergencias y Seguridad 9-1-1.,Proceso adjudicado y celebrado,DOP,991200.00,2026-06-30,20:46:42,Si,No,Servicios,No,No,No,https://comunidad.comprasdominicana.gob.do//Public/Tendering/OpportunityDetail/Index?noticeUID=DO1.NTC.1735775
1,EDENORTE-CCC-PEPU-2026-0026,698,Empresa de Distribución Eléctrica del Norte,Procesos de Excepción,Proveedor único,CONTRATACIÓN DEL SERVICIO DE COBRO AUTOMÁTICO O DOMICILIADO (TRANXA),Proceso con etapa cerrada,USD,18000.00,2026-06-30,18:00:26,No,No,Servicios,No,No,No,https://comunidad.comprasdominicana.gob.do//Public/Tendering/OpportunityDetail/Index?noticeUID=DO1.NTC.1735586
2,CONANI-CCC-PEIN-2026-0005,843,Consejo Nacional para la Niñez y la Adolescencia,Procesos de Excepción,Inmuebles para uso estatal,CONTRATACION DE SERVICIO DE ALQUILER PARA SER UTILIZADO COMO OFICINA MUNICIPAL DE SOSUA.,Sobres abiertos o aperturados,DOP,3509581.66,2026-06-30,18:00:12,No,No,Servicios,No,No,No,https://comunidad.comprasdominicana.gob.do//Public/Tendering/OpportunityDetail/Index?noticeUID=DO1.NTC.1735585
3,ASDE-CCC-CP-2026-0015,615,Ayuntamiento Santo Domingo Este,Comparación de Precios,Proceso ordinario,CONTRATACIÓN DE EMPRESAS O PERSONAS FISICAS PARA LA DOTACION DE AYUDAS TECNICAS A FUNDACIONES ELEGIDAS EN ASAMBLEA POR MUNICIPE CORRESPONDIENTE AL PRESUPUESTO PARTICIPATIVO MUNICIPAL (PPM) AÑO 2026,Proceso con etapa cerrada,DOP,6000000.00,2026-06-30,18:00:12,No,No,Bienes,No,No,No,https://comunidad.comprasdominicana.gob.do//Public/Tendering/OpportunityDetail/Index?noticeUID=DO1.NTC.1735677
4,SIV-DAF-CM-2026-0031,809,Superintendencia de Valores,Contratación Menor,Proceso ordinario,Contratación de servicio de consultoría para el estudio y cálculo actuarial del pasivo laboral de la Superintendencia del Mercado de Valores 2026-2028,Sobres abiertos o aperturados,DOP,1500000.00,2026-06-30,18:00:12,No,No,Servicios,No,No,No,https://comunidad.comprasdominicana.gob.do//Public/Tendering/OpportunityDetail/Index?noticeUID=DO1.NTC.1735272


Column names:
1. CODIGO_PROCESO
2. CODIGO_UNIDAD_COMPRA
3. UNIDAD_COMPRA
4. MODALIDAD
5. TIPO_EXCEPCION
6. CARATULA
7. ESTADO_PROCESO
8. MONEDA
9. MONTO_ESTIMADO
10. FECHA_PUBLICACION
11. HORA_PUBLICACION
12. DIRIGIDO_MIPYMES
13. DIRIGIDO_MIPYMES_MUJERES
14. OBJETO_PROCESO
15. DECRETO_PRESIDENCIAL
16. COMPRA_VERDE
17. COMPRA_CONJUNTA
18. URL


In [4]:
profile = pd.DataFrame({
    "Column": df.columns,
    "Dtype": df.dtypes.astype(str).values,
    "Non_Null": df.notna().sum().values,
    "Null_%": df.isna().mean().mul(100).round(2).values,
    "Unique": df.nunique(dropna=True).values,
})
display(profile)

,Column,Dtype,Non_Null,Null_%,Unique
0,CODIGO_PROCESO,object,610416,0.0,610416
1,CODIGO_UNIDAD_COMPRA,int64,610416,0.0,731
2,UNIDAD_COMPRA,object,610416,0.0,731
3,MODALIDAD,object,610416,0.0,10
4,TIPO_EXCEPCION,object,610416,0.0,14
5,CARATULA,object,610416,0.0,479824
6,ESTADO_PROCESO,object,610416,0.0,10
7,MONEDA,object,610416,0.0,6
8,MONTO_ESTIMADO,float64,610416,0.0,211697
9,FECHA_PUBLICACION,object,610416,0.0,2826


In [5]:
df["FECHA_PUBLICACION_DT"] = pd.to_datetime(df["FECHA_PUBLICACION"], errors="coerce")
print(
    f"Date range: {df['FECHA_PUBLICACION_DT'].min():%Y-%m-%d} → "
    f"{df['FECHA_PUBLICACION_DT'].max():%Y-%m-%d}"
)
print("Currency counts:")
display(df["MONEDA"].value_counts(dropna=False).rename("Records").to_frame())
print("Top modalities:")
display(df["MODALIDAD"].value_counts().head(10).rename("Records").to_frame())

Date range: 2015-03-27 → 2026-06-30
Currency counts:


,Records
MONEDA,
DOP,610104
USD,282
EUR,27
GBP,1
COP,1
MXN,1


Top modalities:


,Records
MODALIDAD,
Compras por Debajo del Umbral,356034
Contratación Menor,166441
Procesos de Excepción,50866
Comparación de Precios,28759
Licitación Pública Nacional,7346
Subasta Inversa,679
Sorteo de Obras,143
Licitación Pública Internacional,102
Licitación Restringida,32


## Data-quality findings

The source contains multiple currencies, negative monetary values, extreme monetary observations, and zero-amount records. The ETL removes negative, null, and extreme amounts while retaining zeros as visible source observations.

No historical FX conversion is applied. Cross-currency financial aggregation is therefore avoided; DOP-only values are used for executive financial KPIs.

In [6]:
MAX_REASONABLE_AMOUNT = 100_000_000_000
amount = pd.to_numeric(df["MONTO_ESTIMADO"], errors="coerce")

quality_summary = pd.DataFrame({
    "Check": [
        "Negative amounts",
        "Zero amounts",
        "Null amounts",
        "Extreme amounts (>=100B)",
    ],
    "Records": [
        int((amount < 0).sum()),
        int((amount == 0).sum()),
        int(amount.isna().sum()),
        int((amount >= MAX_REASONABLE_AMOUNT).sum()),
    ],
})
display(quality_summary)

,Check,Records
0,Negative amounts,9
1,Zero amounts,19
2,Null amounts,0
3,Extreme amounts (>=100B),5


## Apply the production ETL cleaning function

In [7]:
df_for_etl = df.drop(columns=["FECHA_PUBLICACION_DT"], errors="ignore")
df_cleaned = clean_procurement_data(df_for_etl)

print(f"Raw records:    {len(df):,}")
print(f"Clean records:  {len(df_cleaned):,}")
print(f"Removed:        {len(df) - len(df_cleaned):,}")
print(f"Retention:      {len(df_cleaned) / len(df) * 100:.4f}%")
print(f"Zero amounts retained: {(df_cleaned['MONTO_ESTIMADO'] == 0).sum():,}")

[ETL] Cleaning and transforming data...
[ETL] Removed 14 invalid amount rows (negative/null/extreme).
[ETL] Final records: 610,402 | Zero-amount records retained: 19
Raw records:    610,416
Clean records:  610,402
Removed:        14
Retention:      99.9977%
Zero amounts retained: 19


## Data integrity checks

After cleaning, validate the process key and core monetary rules before continuing to the analytical metrics.


In [8]:
# Core post-ETL integrity checks
duplicate_process_codes = int(df_cleaned["CODIGO_PROCESO"].duplicated().sum())
null_process_codes = int(df_cleaned["CODIGO_PROCESO"].isna().sum())
negative_clean_amounts = int((df_cleaned["MONTO_ESTIMADO"] < 0).sum())
null_clean_amounts = int(df_cleaned["MONTO_ESTIMADO"].isna().sum())
extreme_clean_amounts = int((df_cleaned["MONTO_ESTIMADO"] >= MAX_REASONABLE_AMOUNT).sum())

integrity_checks = pd.DataFrame({
    "Check": [
        "Duplicated process codes",
        "Null process codes",
        "Negative cleaned amounts",
        "Null cleaned amounts",
        "Extreme cleaned amounts (>=100B)",
    ],
    "Records": [
        duplicate_process_codes,
        null_process_codes,
        negative_clean_amounts,
        null_clean_amounts,
        extreme_clean_amounts,
    ],
})
display(integrity_checks)

assert duplicate_process_codes == 0, "Duplicate process codes remain after cleaning."
assert null_process_codes == 0, "Null process codes remain after cleaning."
assert negative_clean_amounts == 0, "Negative amounts remain after cleaning."
assert null_clean_amounts == 0, "Null amounts remain after cleaning."
assert extreme_clean_amounts == 0, "Extreme amounts remain after cleaning."
print("All core integrity checks passed.")


,Check,Records
0,Duplicated process codes,0
1,Null process codes,0
2,Negative cleaned amounts,0
3,Null cleaned amounts,0
4,Extreme cleaned amounts (>=100B),0


All core integrity checks passed.


## Raw vs. Cleaned monetary distribution

`describe()` is intentionally kept in the notebook because it provides direct evidence for the cleaning decision. The important comparison is not only the mean: the median and upper quantiles show how the extreme observations affect the raw distribution.

In [9]:
raw_amount = pd.to_numeric(df["MONTO_ESTIMADO"], errors="coerce")
clean_amount = pd.to_numeric(df_cleaned["MONTO_ESTIMADO"], errors="coerce")

raw_describe = raw_amount.describe()
clean_describe = clean_amount.describe()

describe_comparison = pd.concat(
    [raw_describe.rename("Raw Dataset"), clean_describe.rename("Cleaned Dataset")],
    axis=1,
)
display(describe_comparison)

,Raw Dataset,Cleaned Dataset
count,6.104160e+05,6.104020e+05
mean,2.860754e+15,4.909095e+06
std,2.235082e+18,2.032957e+08
min,-1.231477e+09,0.000000e+00
25%,4.500000e+04,4.500000e+04
50%,1.338500e+05,1.338699e+05
75%,4.788392e+05,4.788000e+05
max,1.746250e+21,5.889425e+10


In [10]:
# Human-readable comparison table
describe_table = pd.DataFrame({
    "Metric": ["Count", "Mean", "Std Dev", "Min", "25%", "50% (Median)", "75%", "Max"],
    "Raw Dataset": raw_describe.reindex(["count", "mean", "std", "min", "25%", "50%", "75%", "max"]).values,
    "Cleaned Dataset": clean_describe.reindex(["count", "mean", "std", "min", "25%", "50%", "75%", "max"]).values,
})

# Use plain DataFrame formatting rather than pandas Styler so the notebook does not
# require the optional Jinja2 dependency just to render this table.
for column in ["Raw Dataset", "Cleaned Dataset"]:
    describe_table[column] = describe_table[column].map(
        lambda value: f"{value:,.2f}" if pd.notna(value) else "NaN"
    )

display(describe_table)


,Metric,Raw Dataset,Cleaned Dataset
0,Count,"610,416.00","610,402.00"
1,Mean,"2,860,754,063,936,701.50","4,909,094.93"
2,Std Dev,"2,235,082,019,392,167,168.00","203,295,733.67"
3,Min,"-1,231,476,600.00",0.00
4,25%,"45,000.00","45,000.00"
5,50% (Median),"133,850.00","133,869.95"
6,75%,"478,839.18","478,800.00"
7,Max,"1,746,249,999,999,999,934,464.00","58,894,249,136.00"


## Records removed by the ETL

Each excluded row is classified using the same conditions used by the cleaning process. This makes the transformation auditable instead of treating the removal count as a black box.
> When the repository sample is used, zero removals are possible because the sample is a reproducibility subset and does not necessarily contain the anomalies found in the full source.


In [11]:
# Reconstruct the exact removal logic for auditability.
invalid_mask = (
    pd.to_numeric(df_for_etl["MONTO_ESTIMADO"], errors="coerce").isna()
    | pd.to_numeric(df_for_etl["MONTO_ESTIMADO"], errors="coerce").lt(0)
    | pd.to_numeric(df_for_etl["MONTO_ESTIMADO"], errors="coerce").ge(MAX_REASONABLE_AMOUNT)
)

df_removed = df_for_etl.loc[invalid_mask].copy()
removed_amount = pd.to_numeric(df_removed["MONTO_ESTIMADO"], errors="coerce")

df_removed["Removal_Reason"] = np.select(
    [
        removed_amount.isna(),
        removed_amount < 0,
        removed_amount >= MAX_REASONABLE_AMOUNT,
    ],
    ["Null amount", "Negative amount", "Extreme amount (>=100B)"],
    default="Other",
)

print(f"Total rows removed: {len(df_removed):,}")
display(df_removed[["CODIGO_PROCESO", "MONTO_ESTIMADO", "MONEDA", "Removal_Reason"]].head(20))
display(df_removed["Removal_Reason"].value_counts().rename("Records").to_frame())

Total rows removed: 14


,CODIGO_PROCESO,MONTO_ESTIMADO,MONEDA,Removal_Reason
64204,HRUSVP-DAF-CD-2025-0079,-8.995400e+04,DOP,Negative amount
74890,MEM-DAF-CM-2025-0094,-1.231477e+09,DOP,Negative amount
130954,Inst. Nac. de Cancer-DAF-CM-2024-0068,-1.300000e+04,DOP,Negative amount
263165,CEIZTUR-DAF-CM-2022-0025,-2.814993e+05,DOP,Negative amount
388605,DIGEV-DAF-CM-2020-0153,1.746250e+21,DOP,Extreme amount (>=100B)
420359,MEM-UC-CD-2019-0395,-3.561000e+04,DOP,Negative amount
432110,AYUNTAMIENTO SAN P.-CCC-LPI-2019-0001,2.709320e+12,DOP,Extreme amount (>=100B)
433404,ARS SENASA-CCC-LPN-2019-0014,4.000000e+12,DOP,Extreme amount (>=100B)
479160,MAP-CCC-CP-2019-0006,-4.494200e+05,DOP,Negative amount
514842,MINERD-CCC-CP-2018-0048,-2.044149e+07,DOP,Negative amount


,Records
Removal_Reason,
Negative amount,9
Extreme amount (>=100B),5


## Currency analysis

Amounts remain in their source currencies. They must not be summed together as if they were all DOP. The main financial KPIs therefore use the DOP slice only.

In [12]:
currency_summary = (
    df_cleaned.groupby("MONEDA", dropna=False)["MONTO_ESTIMADO"]
    .agg(
        Records="size",
        Total_Amount="sum",
        Average_Amount="mean",
        Median_Amount="median",
    )
    .reset_index()
    .sort_values("Records", ascending=False)
)
display(currency_summary)

,MONEDA,Records,Total_Amount,Average_Amount,Median_Amount
1,DOP,610090,2.995095e+12,4.909268e+06,133877.04
5,USD,282,1.060742e+09,3.761496e+06,120491.95
2,EUR,27,3.648137e+08,1.351162e+07,6735000.00
0,COP,1,1.852950e+05,1.852950e+05,185295.00
3,GBP,1,2.670000e+05,2.670000e+05,267000.00
4,MXN,1,9.435000e+04,9.435000e+04,94350.00


## Executive KPIs

These are descriptive KPIs for the cleaned dataset. Financial values are restricted to `DOP` to avoid unsupported historical FX assumptions.

In [13]:
total_records = len(df_cleaned)
dop = df_cleaned.loc[df_cleaned["MONEDA"] == "DOP"].copy()

dop_process_count = len(dop)
dop_record_share = dop_process_count / total_records if total_records else 0
dop_total_amount = dop["MONTO_ESTIMADO"].sum()
dop_average_amount = dop["MONTO_ESTIMADO"].mean()
dop_median_amount = dop["MONTO_ESTIMADO"].median()
mean_median_ratio = (
    dop_average_amount / dop_median_amount
    if pd.notna(dop_median_amount) and dop_median_amount != 0
    else np.nan
)
etl_retention = len(df_cleaned) / len(df) if len(df) else 0

executive_kpis = pd.DataFrame({
    "Metric": [
        "Cleaned records",
        "DOP records",
        "DOP record share",
        "Total DOP estimated amount",
        "Average DOP estimated amount",
        "Median DOP estimated amount",
        "DOP mean / median ratio",
        "ETL retention",
    ],
    "Value": [
        total_records,
        dop_process_count,
        dop_record_share,
        dop_total_amount,
        dop_average_amount,
        dop_median_amount,
        mean_median_ratio,
        etl_retention,
    ],
})
display(executive_kpis)

print("DOP-only executive KPI reconciliation")
print(f"DOP process count:      {dop_process_count:,}")
print(f"DOP record share:       {dop_record_share:.2%}")
print(f"Total DOP amount:       RD$ {dop_total_amount:,.2f}")
print(f"Average DOP amount:     RD$ {dop_average_amount:,.2f}")
print(f"Median DOP amount:      RD$ {dop_median_amount:,.2f}")
print(f"Mean / median ratio:    {mean_median_ratio:.2f}x")

,Metric,Value
0,Cleaned records,6.104020e+05
1,DOP records,6.100900e+05
2,DOP record share,9.994889e-01
3,Total DOP estimated amount,2.995095e+12
4,Average DOP estimated amount,4.909268e+06
5,Median DOP estimated amount,1.338770e+05
6,DOP mean / median ratio,3.666998e+01
7,ETL retention,9.999771e-01


DOP-only executive KPI reconciliation
DOP process count:      610,090
DOP record share:       99.95%
Total DOP amount:       RD$ 2,995,095,263,307.73
Average DOP amount:     RD$ 4,909,267.92
Median DOP amount:      RD$ 133,877.04
Mean / median ratio:    36.67x


## Procurement profile

The cleaned dataset also contains operational and source-reported policy flags that can be summarized without implying award outcomes or causal effects.

In [14]:
purchasing_units = df_cleaned["CODIGO_UNIDAD_COMPRA"].nunique(dropna=True)
modalities = df_cleaned["MODALIDAD"].nunique(dropna=True)
statuses = df_cleaned["ESTADO_PROCESO"].nunique(dropna=True)

mipyme_records = df_cleaned["DIRIGIDO_MIPYMES"].eq(True).sum()
mipyme_women_records = df_cleaned["DIRIGIDO_MIPYMES_MUJERES"].eq(True).sum()
green_records = df_cleaned["COMPRA_VERDE"].eq(True).sum()
joint_records = df_cleaned["COMPRA_CONJUNTA"].eq(True).sum()

profile_metrics = pd.DataFrame({
    "Metric": [
        "Purchasing units",
        "Procurement modalities",
        "Process statuses",
        "Processes marked for MiPyMEs",
        "Processes marked for MiPyMEs women",
        "Green-purchase records",
        "Joint-purchase records",
    ],
    "Records / Count": [
        purchasing_units,
        modalities,
        statuses,
        mipyme_records,
        mipyme_women_records,
        green_records,
        joint_records,
    ],
})
display(profile_metrics)

print(f"MiPyMEs share:            {mipyme_records / total_records:.2%}")
print(f"MiPyMEs women share:      {mipyme_women_records / total_records:.2%}")
print(f"Green purchase share:     {green_records / total_records:.2%}")
print(f"Joint purchase share:     {joint_records / total_records:.2%}")

,Metric,Records / Count
0,Purchasing units,731
1,Procurement modalities,10
2,Process statuses,10
3,Processes marked for MiPyMEs,58522
4,Processes marked for MiPyMEs women,10114
5,Green-purchase records,3083
6,Joint-purchase records,102


MiPyMEs share:            9.59%
MiPyMEs women share:      1.66%
Green purchase share:     0.51%
Joint purchase share:     0.02%


In [15]:
# Modality concentration by process count
modality_counts = df_cleaned["MODALIDAD"].value_counts(dropna=False)
top_3_share = modality_counts.head(3).sum() / total_records if total_records else 0

print("Top 3 modalities by process count:")
display(modality_counts.head(3).rename("Records").to_frame())
print(f"Top 3 modalities share: {top_3_share:.2%}")

Top 3 modalities by process count:


,Records
MODALIDAD,
Compras por Debajo del Umbral,356031
Contratación Menor,166437
Procesos de Excepción,50864


Top 3 modalities share: 93.93%


## Optional: reconcile with the persisted cleaned CSV

When the full raw dataset and its persisted processed CSV are both present locally, this cell verifies that the notebook's in-memory ETL output agrees with the saved file.

In [16]:
if PROCESSED_PATH.exists() and DATA_PATH == RAW_PATH:
    df_processed = pd.read_csv(PROCESSED_PATH, low_memory=False)
    print(f"Persisted processed rows: {len(df_processed):,}")
    print(f"Notebook-cleaned rows:    {len(df_cleaned):,}")
    print(f"Same row count:           {len(df_processed) == len(df_cleaned)}")

    compare_columns = [
        "CODIGO_PROCESO",
        "MONTO_ESTIMADO",
        "MONEDA",
        "YEAR",
        "MONTH",
        "QUARTER",
    ]
    pd.testing.assert_frame_equal(
        df_processed[compare_columns].reset_index(drop=True),
        df_cleaned[compare_columns].reset_index(drop=True),
        check_dtype=False,
        check_names=True,
    )
    print("Key-column equality:      True")
else:
    print("No full persisted cleaned CSV is available; the notebook is using the in-memory ETL result.")

No full persisted cleaned CSV is available; the notebook is using the in-memory ETL result.


## Why these metrics matter

- **Raw vs. cleaned `describe()`** shows the effect of extreme observations rather than hiding the cleaning impact.
- **Retention and removal counts** make the ETL decision measurable.
- **DOP share and DOP-only monetary KPIs** prevent unsupported cross-currency aggregation.
- **Mean vs. median** explains the skewed distribution.
- **MiPyMEs, women, green and joint-purchase shares** quantify fields that can support descriptive business questions.
- **Purchasing units, modalities and statuses** show the dimensional coverage available to the SQL and Power BI layers.

## Final EDA conclusions

1. The raw monetary distribution contains extreme values that disproportionately affect the mean.
2. The ETL removes negative, null and extreme monetary records while retaining zero-valued observations.
3. The median is much more stable than the raw mean under the observed outliers, which supports using both statistics rather than relying on the average alone.
4. Currency is preserved as source data; no unsupported FX conversion is introduced.
5. DOP-only financial KPIs can be reconciled between the notebook, SQL layer and Power BI.
6. The cleaned dataset is ready for SQL Server staging, dimensional modeling and reporting.